In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

llm = ChatOpenAI()

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!",
 'explanation': "This joke plays on the idea of a pizza needing to see a therapist because it is overwhelmed by all of its toppings. In reality, pizzas don't have feelings or emotions, so the image of a pizza going to therapy is humorous because it is so absurd. The punchline adds to the humor by referencing the problem of the toppings causing the pizza to fall apart, as if it has emotional issues like a person would. Overall, the joke is a lighthearted and silly way to imagine a pizza seeking help for a completely non-problem."}

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!", 'explanation': "This joke plays on the idea of a pizza needing to see a therapist because it is overwhelmed by all of its toppings. In reality, pizzas don't have feelings or emotions, so the image of a pizza going to therapy is humorous because it is so absurd. The punchline adds to the humor by referencing the problem of the toppings causing the pizza to fall apart, as if it has emotional issues like a person would. Overall, the joke is a lighthearted and silly way to imagine a pizza seeking help for a completely non-problem."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18e3e1-9365-6dde-8002-89031d9ed5d0'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-02T06:47:58.730577+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': ''

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!", 'explanation': "This joke plays on the idea of a pizza needing to see a therapist because it is overwhelmed by all of its toppings. In reality, pizzas don't have feelings or emotions, so the image of a pizza going to therapy is humorous because it is so absurd. The punchline adds to the humor by referencing the problem of the toppings causing the pizza to fall apart, as if it has emotional issues like a person would. Overall, the joke is a lighthearted and silly way to imagine a pizza seeking help for a completely non-problem."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18e3e1-9365-6dde-8002-89031d9ed5d0'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-02T06:47:58.730577+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '

In [10]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta go to the party? \nBecause it heard it was going to be a saucy affair!',
 'explanation': 'This joke is a play on words, using the term "saucy" in two different contexts. In the literal sense, pasta dishes are often served with sauce. So, when the pasta heard it was going to be a "saucy affair," it thought it would be a good time to join the party because it would be a fun and exciting event. The humor comes from the double meaning of "saucy" referring to both the pasta sauce and the lively atmosphere of the party.'}

In [11]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!", 'explanation': "This joke plays on the idea of a pizza needing to see a therapist because it is overwhelmed by all of its toppings. In reality, pizzas don't have feelings or emotions, so the image of a pizza going to therapy is humorous because it is so absurd. The punchline adds to the humor by referencing the problem of the toppings causing the pizza to fall apart, as if it has emotional issues like a person would. Overall, the joke is a lighthearted and silly way to imagine a pizza seeking help for a completely non-problem."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18e3e1-9365-6dde-8002-89031d9ed5d0'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-02T06:47:58.730577+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': ''

In [12]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!", 'explanation': "This joke plays on the idea of a pizza needing to see a therapist because it is overwhelmed by all of its toppings. In reality, pizzas don't have feelings or emotions, so the image of a pizza going to therapy is humorous because it is so absurd. The punchline adds to the humor by referencing the problem of the toppings causing the pizza to fall apart, as if it has emotional issues like a person would. Overall, the joke is a lighthearted and silly way to imagine a pizza seeking help for a completely non-problem."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18e3e1-9365-6dde-8002-89031d9ed5d0'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-02T06:47:58.730577+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '

Time Travel

In [15]:

workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [16]:

workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f18e3e1-801e-608c-8001-c73fff941158"}})

{'topic': 'pizza',
 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!",
 'explanation': 'This joke plays on the idea of a pizza "going" to therapy as if it were a person seeking help for a personal issue. The punchline reveals that the pizza\'s reason for seeking therapy is because it had too many toppings, causing it to fall apart or not hold together properly. It is a play on words, as holding together can refer to both the physical structure of the pizza and the mental/emotional state of a person seeking therapy.'}

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!", 'explanation': 'This joke plays on the idea of a pizza "going" to therapy as if it were a person seeking help for a personal issue. The punchline reveals that the pizza\'s reason for seeking therapy is because it had too many toppings, causing it to fall apart or not hold together properly. It is a play on words, as holding together can refer to both the physical structure of the pizza and the mental/emotional state of a person seeking therapy.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18e3f9-44e5-689a-8003-e985df85c40c'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-08-02T06:58:34.744180+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18e3f9-16bb-6e88-8002-68f51c486727'}}, tasks=(), interrupts=(

Updating_state

In [18]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f18e3fa-a104-667a-8000-abd8b9835b0c'}}

In [19]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18e3fa-a104-667a-8000-abd8b9835b0c'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-08-02T06:59:11.247312+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='01c42eb5-bbdc-6a09-8ad0-43c035f2df8f', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!", 'explanation': 'This joke plays on the idea of a pizza "going" to therapy as if it were a person seeking help for a personal issue. The punchline reveals that the pizza\'s reason for seeking therapy is because it had t

In [22]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f18e3fa-a104-667a-8000-abd8b9835b0c"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to the doctor? Because it was feeling a little flat!',
 'explanation': 'This joke plays on the double meaning of the word "flat." In one sense, "flat" can refer to the physical characteristic of being flattened or compressed. In this context, the samosa is feeling "flat" because it has lost its usual plump and triangular shape. On the other hand, "feeling flat" can also mean feeling unwell or not quite oneself. So in the joke, the samosa goes to the doctor because it is physically flat and also feeling unwell.'}

In [23]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to the doctor? Because it was feeling a little flat!', 'explanation': 'This joke plays on the double meaning of the word "flat." In one sense, "flat" can refer to the physical characteristic of being flattened or compressed. In this context, the samosa is feeling "flat" because it has lost its usual plump and triangular shape. On the other hand, "feeling flat" can also mean feeling unwell or not quite oneself. So in the joke, the samosa goes to the doctor because it is physically flat and also feeling unwell.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18e3fe-9059-6e5c-8002-3d30c2a63311'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-02T07:00:56.873877+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18e3fe-78a0-6b30-8001-598590c8de78'}}, tasks=(), interrupts=()),
 StateSnapshot(valu